cluster the residual (non-government, non-candidate) ad bodies into latent topics with LDA so we can filter out commercial/spam topics (fashion, fitness apps, retail) from the political-adjacent subset, see what kinds of political messaging are running outside party/candidate channels (climate, cost-of-living, voice, housing, etc), and bake topic labels into v3 parquet for cross-tabbing with sentiment (nb 04) and spend/impressions.

plan: fit a single LDA on the full residual corpus, eyeball the top words per topic, hand-label each topic in a csv, join the labels back to the corpus.

this notebook walks through the exploration - the attrition funnel, the body-dropout diagnostics, the k-sweep that justifies k=20, and the final k=20 console review. the production pipeline (LDA fit, write topic_terms.csv, write intermediate parquet) lives in 03_topics.py. nothing gets written from this notebook.

preprocessing — load v2 parquet, one row per ad (ad_seq_no = 1) and non-classified (match_type IS NULL). pull the first creative body, tokenise with RegexTokenizer, drop stop words (english defaults + a bit of domain noise). vectorise word counts with CountVectorizer — raw counts, not TF-IDF, because LDA needs integer term frequencies.

cache the vectorised features so the k-sweep doesnt re-run preprocessing per k.

spark session.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    array_contains, col, coalesce, concat_ws, expr, length, lit,
)

# Same parquet-committer override as notebook 02 — cluster default points at an EMR
# class whose JAR isn't on the classpath.
spark = SparkSession.builder \
    .appName('FB_API_topics') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

Master: yarn
Spark version: 3.5.0


26/05/14 02:25:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


paths.

In [2]:
V2_PATH = '/user/s3348393/main/preprocessing/v2/parquet'

load v2 and filter to the residual corpus. keep one row per ad (ad_seq_no = 1) and ads that didnt classify as candidate/party/government (match_type IS NULL), are english (array_contains('languages', 'en')) which drops non-english clusters, and arent from a byline we've explicitly tagged as commercial/non-political (COMMERCIAL_BYLINES) — that set grows iteratively as LDA surfaces noise topics.

body-text extraction is more generous than just creative_bodies[0]. first_non_empty(arr) returns the first non-null, non-empty element of an array column — recovers ads where the first body entry is null but a later one is real (multi-variant ads), or where the singular ad_creative_body was null but other text fields are populated. then concat creative_bodies + creative_link_descs + creative_link_titles into a single doc — more text = better LDA signal, and ads with no body but a populated link description ("sign the petition", "donate now") still contribute. drop ads with no text in any of these fields — LDA cant deal with empty docs.

creative_link_captions is skipped — usually just a domain name (noise).

In [3]:
df = spark.read.parquet(V2_PATH)
print('All v2 rows:    ', df.count())


def first_non_empty(col_name):
    """First non-null, non-empty element of an array column. Returns null if none."""
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")


# Bylines we've identified as clearly non-political (commercial, recruitment, etc.)
# and want excluded from the LDA corpus. Iteratively grown — add bylines here when
# LDA surfaces them as dominating a noise/commercial topic.
COMMERCIAL_BYLINES = {
    'Access',                            # Indigenous Employment Australia — job listings
    'Streamotion Pty Ltd',               # Kayo / Binge sports + entertainment streaming
    'SBS Australia',                     # SBS On Demand streaming promotions
    'SBS Arabic24',                      # SBS language-stream marketing
    'SBS Mandarin中文普通话',            # SBS language-stream marketing
    'The Squiz',                         # paid news newsletter
    'Hair Cooki'                           #hair care ad
}


corpus = df.filter(
        (col('ad_seq_no') == 1) &
        col('match_type').isNull() &
        # Language: keep ads where languages is null (untagged — ~96% of residual)
        # OR explicitly tagged as English. Drops the small confirmed-non-English tail.
        (col('languages').isNull() | array_contains('languages', 'en')) &
        ~col('bylines').isin(list(COMMERCIAL_BYLINES))
    ) \
    .withColumn('body_text',  first_non_empty('creative_bodies')) \
    .withColumn('desc_text',  first_non_empty('creative_link_descs')) \
    .withColumn('title_text', first_non_empty('creative_link_titles')) \
    .withColumn('body',
        concat_ws(' ',
            coalesce(col('body_text'),  lit('')),
            coalesce(col('desc_text'),  lit('')),
            coalesce(col('title_text'), lit('')),
        )
    ) \
    .filter(length(col('body')) > 0) \
    .drop('body_text', 'desc_text', 'title_text')

print('Residual corpus:', corpus.count())
corpus.select('page_name', 'bylines', 'body').show(3, truncate=80)

All v2 rows:     5796491


Residual corpus: 96286
+---------------------------------+---------------------------------+--------------------------------------------------------------------------------+
|                        page_name|                          bylines|                                                                            body|
+---------------------------------+---------------------------------+--------------------------------------------------------------------------------+
|    Australian Ethical Investment|    Australian Ethical Investment|Creating sustainable impact or getting top returns? Por que no los dos?! 🤷\n...|
|Australian Automobile Association|Australian Automobile Association|We’re all starting to move again, but we need to get to our destination effic...|
|     Greenpeace Australia Pacific|     Greenpeace Australia Pacific|DID YOU KNOW that if global temperatures warm to 🔥 2.0 degrees above pre-ind...|
+---------------------------------+---------------------------------+----

attrition diagnostic — why's the residual corpus only 96k?

5.8M v2 rows collapse to ~96k after the filters above. most of that drop is structural (ad_seq_no = 1 deduplicates multi-row ads, match_type IS NULL drops the political-classified majority), but the last step — length(body) > 0 — is the one to double-check: it drops every ad with no usable creative text, and the hunch is that those are mostly image- or video-only ads.

two diagnostics. funnel — apply each filter cumulatively, show how many rows survive each step. tells us which filter is doing the work. then body-dropout breakdown — of ads that pass every other filter but get dropped for no text in creative_bodies + creative_link_descs + creative_link_titles, count how many also have no text in creative_link_captions (so no creative text anywhere — image/video-only) vs caption-only. plus a random sample of ad_snapshot_urls to eyeball in a browser.

In [4]:
# Attrition funnel — apply each section-1.3 filter cumulatively, see what survives.
total = df.count()

f_seq  = (col('ad_seq_no') == 1)
f_mt   = col('match_type').isNull()
f_lang = (col('languages').isNull() | array_contains('languages', 'en'))
f_byl  = ~col('bylines').isin(list(COMMERCIAL_BYLINES))

n_seq  = df.filter(f_seq).count()
n_mt   = df.filter(f_seq & f_mt).count()
n_lang = df.filter(f_seq & f_mt & f_lang).count()
n_byl  = df.filter(f_seq & f_mt & f_lang & f_byl).count()
n_body = corpus.count()  # final corpus from cell above

print(f'{"step":<38}{"rows":>12}  {"% of v2":>8}')
print('-' * 60)
for label, n in [
    ('all v2 rows',                       total),
    ('+ ad_seq_no == 1',                  n_seq),
    ('+ match_type IS NULL',              n_mt),
    ('+ english or untagged languages',   n_lang),
    ('+ exclude commercial bylines',      n_byl),
    ('+ length(body) > 0 (final corpus)', n_body),
]:
    print(f'{label:<38}{n:>12,}  {n/total:>7.1%}')

print()
print(f'Dropped specifically by length(body) > 0: {n_byl - n_body:,}')

step                                          rows   % of v2
------------------------------------------------------------
all v2 rows                              5,796,491   100.0%
+ ad_seq_no == 1                           151,578     2.6%
+ match_type IS NULL                       105,814     1.8%
+ english or untagged languages            104,547     1.8%
+ exclude commercial bylines                96,286     1.7%
+ length(body) > 0 (final corpus)           96,286     1.7%

Dropped specifically by length(body) > 0: 0


In [5]:
# Body-dropout breakdown — investigate the ads dropped purely by length(body) > 0.
# These pass ad_seq_no, match_type, language, and commercial-byline filters but have
# no extractable text in creative_bodies/descs/titles.

pre_body = df.filter(f_seq & f_mt & f_lang & f_byl) \
    .withColumn('body_text',    first_non_empty('creative_bodies')) \
    .withColumn('desc_text',    first_non_empty('creative_link_descs')) \
    .withColumn('title_text',   first_non_empty('creative_link_titles')) \
    .withColumn('caption_text', first_non_empty('creative_link_captions'))

is_empty = lambda c: col(c).isNull() | (length(col(c)) == 0)

dropped = pre_body.filter(is_empty('body_text') & is_empty('desc_text') & is_empty('title_text'))
dropped_n = dropped.count()
print(f'Total dropped by body-length filter: {dropped_n:,}\n')

# Bucket the dropouts by what creative text (if any) they DO have.
buckets = dropped.withColumn('bucket',
    expr("""
        case
            when caption_text is null or length(caption_text) = 0
                then 'no_text_anywhere (likely image/video-only)'
            else 'caption_only (usually just a domain)'
        end
    """)
)
buckets.groupBy('bucket').count().orderBy(col('count').desc()).show(truncate=False)

# Random sample of dropped ads — open ad_snapshot_url in a browser to eyeball
# what they actually are. seed=42 keeps the sample stable across reruns.
print('Random sample of dropped ads (open ad_snapshot_url to verify):')
dropped.select('page_name', 'bylines', 'caption_text', 'ad_snapshot_url') \
    .sample(False, 0.001, seed=42) \
    .limit(10) \
    .show(truncate=80)

Total dropped by body-length filter: 802



+------------------------------------------+-----+
|bucket                                    |count|
+------------------------------------------+-----+
|caption_only (usually just a domain)      |403  |
|no_text_anywhere (likely image/video-only)|399  |
+------------------------------------------+-----+

Random sample of dropped ads (open ad_snapshot_url to verify):


+---------------------------+----------------------------+---------------------------------+--------------------------------------------------------------------------------+
|                  page_name|                     bylines|                     caption_text|                                                                 ad_snapshot_url|
+---------------------------+----------------------------+---------------------------------+--------------------------------------------------------------------------------+
|   What can I do? Australia|What Can I Do? Australia Inc|whatcanidoaustralia.thinkific.com|https://www.facebook.com/ads/archive/render_ad/?id=2295039480665247&access_to...|
|The University of Melbourne| The University of Melbourne|              www.vaxfacts.org.au|https://www.facebook.com/ads/archive/render_ad/?id=688561895465132&access_tok...|
+---------------------------+----------------------------+---------------------------------+--------------------------------------

stop words. english defaults plus a small list of domain noise: URL fragments, generic call-to-action words. kept deliberately short — minDF and maxDF in CountVectorizer handle most frequency-based filtering for us. iterate after the first LDA fit if specific tokens are dominating topics with no signal.

In [6]:
from pyspark.ml.feature import StopWordsRemover

stop_words = StopWordsRemover.loadDefaultStopWords('english') + [
    # URL / web junk that survives tokenisation
    'https', 'http', 'www', 'com', 'org', 'au', 'co', 'html',
    # contraction fragments surviving minTokenLength=2
    're', 've', 'll',
    # generic fillers (high frequency, low topic-discrimination value)
    'help', 'time', 'like', 'need', 'make', 'take', 'people',
    'year', 'years', 'today', 'also', 'will', 'can', 'get',
    'see', 'know', 'one', 'two', 'new', 'now', 'us',
    # generic CTA (kept short — don't strip 'petition', 'donate', 'sign', etc.
    # since those carry topic signal)
    'click', 'learn',
]

print('Stop-words list size:', len(stop_words))

Stop-words list size: 215


preprocessing pipeline. three stages: RegexTokenizer (split on \W+, lowercase, drop tokens shorter than 2 chars) then StopWordsRemover (english defaults + domain noise) then CountVectorizer (vocab≤5,000, term must appear in ≥100 ads, term must appear in ≤30% of ads).

minDF=100 is tighter than the usual default — squashes the long-tail vocab used by individual small commercial advertisers and forces LDA to cluster on vocabulary actually shared across the political-advocacy ecosystem. niche-but-real political terms (woodside, quoll, uyghur) should still clear the threshold; one-off commercial product names wont.

wrapping in a Pipeline so we get .fit().transform() in one go and a single fitted artefact to poke at afterwards.

In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, CountVectorizer

tokenizer = RegexTokenizer(
    inputCol='body', outputCol='raw_tokens',
    pattern=r'\W+', toLowercase=True, minTokenLength=2,
)

remover = StopWordsRemover(
    inputCol='raw_tokens', outputCol='tokens',
    stopWords=stop_words,
)

vectorizer = CountVectorizer(
    inputCol='tokens', outputCol='features',
    vocabSize=5000,
    minDF=100,    # term must appear in ≥100 ads — bumped from 50 to compress long-tail vocab
    maxDF=0.3,    # term appearing in >30% of ads is dropped (auto stop-word filter)
)

prep_pipeline = Pipeline(stages=[tokenizer, remover, vectorizer])

fit, transform, cache. fit the pipeline once. the output features_df carries every original column plus raw_tokens, tokens, and features (the sparse count vector LDA consumes). cache it so the k-sweep below doesnt re-run preprocessing per k.

the .count() call forces spark to actually materialise the cache — without it the cache is registered lazily and nothing happens until something else triggers an action.

In [8]:
prep_model  = prep_pipeline.fit(corpus)
features_df = prep_model.transform(corpus).cache()

print('Cached rows:    ', features_df.count())

vocab = prep_model.stages[-1].vocabulary
print('Vocabulary size:', len(vocab))
print('\nTop 30 vocabulary terms (most frequent first):')
print(vocab[:30])

26/05/14 02:25:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Cached rows:     96286
Vocabulary size: 4318

Top 30 vocabulary terms (most frequent first):
['sign', 'australia', 'climate', 'government', 'support', 'community', 'petition', 'australian', 'change', 'vote', 'world', 'protect', 'women', 'future', 'action', 'local', 'free', 'join', 'election', 'stop', 'share', 'council', 'woodside', 'children', 'labor', 'life', 'donate', 'every', 'energy', 'gas']


explore k on a sample. fit LDA at several k values (5, 10, 15, 20) on a 10% sample of the cached features. print top-12 words per topic for each k. eyeball the printouts and pick a k where topics are distinct and each list reads as a coherent theme.

fix seed=42 so comparing k=10 vs k=15 isnt muddled by random init differences. cheap and disposable — no parquet writes here.

In [9]:
from pyspark.ml.clustering import LDA

# 10% sample of the cached features. Cache the sample too — the four LDA fits
# below will scan it repeatedly. seed=42 keeps the sample composition stable.
sample_df = features_df.sample(0.1, seed=42).cache()
print(f'Sample size: {sample_df.count():,}')

# Lookup from CountVectorizer integer term indices back to readable words.
vocab = prep_model.stages[-1].vocabulary

# Fit LDA at each k. seed=42 fixed so k=5 vs k=10 etc. are comparable.
for k in [5, 10, 15, 20]:
    print(f'\n=== k = {k} ===')
    lda = LDA(featuresCol='features', k=k, maxIter=20, seed=42)
    model = lda.fit(sample_df)
    topics = model.describeTopics(maxTermsPerTopic=12).collect()
    for row in topics:
        words = ' '.join(vocab[i] for i in row.termIndices)
        print(f'  Topic {row.topic:>2}: {words}')

sample_df.unpersist()

Sample size: 9,663

=== k = 5 ===


26/05/14 02:26:00 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: sign woodside share australia tell gas petition climate oceans stop ocean australian
  Topic  1: women support violence 2022 council community program school education club live change
  Topic  2: government australia support community sign climate vote local australian petition election join
  Topic  3: climate energy power renewable care labor council change australia community government coal
  Topic  4: early australia learning protect future childcare gift christmas kids children sign world

=== k = 10 ===


26/05/14 02:26:15 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: climate energy australia sign woodside gas tell share solar enough power action
  Topic  1: club festival program live support fodi22 book festivalofdangerousideas flood emergency sept 15
  Topic  2: join super abc australian sign cost health petition medicines australia children product
  Topic  3: agl zero march care coal 2022 climate sign net show qantas sydney
  Topic  4: vaccine reef australians choice freedom show christmas vaxxed fair sign deserve barrier
  Topic  5: future change climate support education vote school life work world australia care
  Topic  6: sign women petition government early children support plastic use australian australia vote
  Topic  7: community government local labor election council australia city state support vote federal
  Topic  8: free stop police life send message text case defence death ants legal
  Topic  9: sign protect petition oceans nature ocean save australia world global wildlife donate

=== k = 15 ===


26/05/14 02:26:27 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: climate australia solar electricity clean toyota transport armenian electric car australians sign
  Topic  1: violence sleep emergency vinnies homelessness flood domestic support place 64 lismore safe
  Topic  2: super medicines join cost switch ethical funds prescription visit choose bank performance
  Topic  3: agl coal care abuse sexual medical without child teams dirty provide banks
  Topic  4: festival live program fodi22 festivalofdangerousideas book multipack sept forests native logging 17
  Topic  5: climate energy change future school education renewable community action australia donation tax
  Topic  6: sign woodside early plastic gas learning women tell use share project petition
  Topic  7: first home research coffee auspol support day independent federal money corruption quiz
  Topic  8: police life ants case send rocky text message visa defence convicted stop
  Topic  9: protect donate sign women nature ukraine gift free future species world save
  Topic 10: 

26/05/14 02:26:40 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


  Topic  0: australia toyota armenian transport clean car electric vehicles climate australians media anc
  Topic  1: violence support emergency place safe women domestic sleep homelessness vinnies flood 64
  Topic  2: super join switch ethical bank performance product information minutes future consider australian
  Topic  3: agl electricity coal care dirty switching green switch climate teams abuse sexual
  Topic  4: reef program barrier great festival fodi22 live forests festivalofdangerousideas multipack book sept
  Topic  5: school education kids schools genus change life donation sustainability students girls future
  Topic  6: sign woodside early climate gas plastic learning tell women share petition use
  Topic  7: coffee home anti corruption rate peninsula auspol mornington frankston body daniel first
  Topic  8: life ants rocky places visa stop death deforestation australia send convicted case
  Topic  9: protect donate nature gift ukraine future species save sign habitat wil

DataFrame[id: string, page_id: string, page_name: string, snapshot_date: date, ad_creation_date: date, ad_delivery_start_date: date, ad_delivery_stop_date: date, creative_bodies: array<string>, creative_link_captions: array<string>, creative_link_descs: array<string>, creative_link_titles: array<string>, currency: string, languages: array<string>, publisher_platforms: array<string>, demographic_distribution: array<struct<age:string,gender:string,percentage:string>>, delivery_by_region: array<struct<percentage:string,region:string>>, ad_snapshot_url: string, bylines: string, spend_lower_bound: bigint, spend_upper_bound: bigint, spend_mid: double, impressions_lower_bound: bigint, impressions_upper_bound: bigint, impressions_mid: double, audience_size_lower_bound: bigint, audience_size_upper_bound: bigint, audience_size_mid: double, ad_seq_no: int, match_type: string, political_party: string, body: string, raw_tokens: array<string>, tokens: array<string>, features: vector]

final fit on the full corpus. refit LDA at k=20 on the full cached features (no sampling). transform the corpus to stick topicDistribution (length-20 vector) and topic_id (argmax) onto every ad. then write two artefacts.

intermediate parquet — full corpus + LDA columns. expensive to recompute (LDA on 50k docs at k=20 takes ~30–60s); write once so the labelling round-trip doesnt refit.

data/topic_terms.csv — one row per topic with top_terms (top 15 words) and top_bylines (top 5 advertisers by ad count). the bylines column is the killer labelling aid: combined with the term list you can usually tell whether a topic is e.g. climate council reef campaign (top byline: climate council) vs greenpeace anti-woodside (top byline: greenpeace) vs commercial residue (top byline: a brand) — way faster than guessing from terms alone.

the csv also has an empty label column for you to fill in. save as data/topic_labels.csv when done.

In [10]:
from pyspark.ml.clustering import LDA
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

K = 20

# Fit LDA at k=K on the full cached features. seed=42 for reproducibility.
lda = LDA(featuresCol='features', k=K, maxIter=20, seed=42)
lda_model = lda.fit(features_df)

# Transform: attach topicDistribution and dominant topic_id to every ad.
classified = lda_model.transform(features_df) \
    .withColumn('topic_array', vector_to_array('topicDistribution')) \
    .withColumn('topic_id', expr('array_position(topic_array, array_max(topic_array)) - 1'))

# Vocabulary for term-index -> word translation.
vocab = prep_model.stages[-1].vocabulary

# Top 15 terms per topic from describeTopics.
topics_rows = lda_model.describeTopics(maxTermsPerTopic=15).collect()
topic_terms = {row.topic: [vocab[i] for i in row.termIndices] for row in topics_rows}

# Top 5 bylines per topic — Spark window function over (topic_id, bylines, count).
w = Window.partitionBy('topic_id').orderBy(desc('count'))
top_bylines_rows = classified.filter(col('bylines').isNotNull()) \
    .groupBy('topic_id', 'bylines').count() \
    .withColumn('rank', row_number().over(w)) \
    .filter(col('rank') <= 5) \
    .orderBy('topic_id', 'rank') \
    .collect()

top_bylines = {}
for row in top_bylines_rows:
    top_bylines.setdefault(row.topic_id, []).append(row.bylines)

# Print to console — easy to scan while you draft labels.
print(f'k = {K}\n')
for tid in range(K):
    words = ' '.join(topic_terms.get(tid, []))
    bls   = ' | '.join(top_bylines.get(tid, []))
    print(f'Topic {tid:>2}:')
    print(f'  Terms:   {words}')
    print(f'  Bylines: {bls}')
    print()

26/05/14 02:26:54 WARN OnlineLDAOptimizer: The input data is not directly cached, which may hurt performance if its parent RDDs are also uncached.


k = 20

Topic  0:
  Terms:   climate oceans sign action ocean global treaty change australia emissions world crisis petition community leaders
  Bylines: Greenpeace Australia Pacific | The Climate Council | Australian Conservation Foundation | Kua | Comms Declare Inc

Topic  1:
  Terms:   festival program live book fodi22 sydney festivalofdangerousideas australia artist 15 sept multipack 17 18 save
  Bylines: Festival of Dangerous Ideas | SBS Cantonese 廣東話節目 | Sydney Opera House | Special Broadcasting Service | Solutions for Australia

Topic  2:
  Terms:   woodside gas project stop sign tell australia whales enough message send drilling share toxic ceo
  Bylines: Greenpeace Australia Pacific | Bank Australia | Amnesty International Australia | The Wilderness Society | World Animal Protection Australia

Topic  3:
  Terms:   coal electricity agl sign switch climate green energy australia renewable power join women china clean
  Bylines: Greenpeace Australia Pacific | Australian Ethical I

that's the LDA story. k=20 picked from the sweep above, topics broadly cover climate, cost-of-living, plastic, ocean, refugees, vaccines, education, electoral and a handful of commercial / festival residue. the production pipeline (LDA fit, write topic_terms.csv, write intermediate parquet) lives in 03_topics.py.

once 03_topics.py has written data/topic_terms.csv, fill in the label and category columns by hand in a spreadsheet - categories like climate, humanitarian_rights, political_advocacy, cost_of_living, noise. save the result as data/topic_labels.csv. nb 04 picks it up from there.